# Exploración inicial de datos con Altair (Vega-Lite)

Notebook preliminar para: **1)** cargar los datos, **2)** hacer una exploración inicial, **3)** exportar datos y gráficos, **4)** reproducir un gráfico de líneas de *precio promedio por año y símbolo*.

> Por defecto usa el dataset de ejemplo `stocks` de `vega_datasets` (es el que genera un gráfico como el de tu imagen). Para usar tus datos, cambia `FUENTE` en la sección 1.

Instalación (una vez): `pip install altair vega_datasets pandas pyarrow vl-convert-python`

In [34]:
import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.disable_max_rows()   # evita el límite de 5.000 filas de Altair
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

OUT = Path('salida'); OUT.mkdir(exist_ok=True)   # carpeta de exportación
print('Altair', alt.__version__)

Altair 5.5.0


## 1. Carga de datos

In [2]:
from pathlib import Path
import pandas as pd

# Opciones: 'demo' (vega_datasets.stocks) o la ruta a tu archivo (.csv, .xlsx, .parquet, .json)
FUENTE = "/content/sample_data/BASE DE DATOS(Hoja1).csv"

def cargar(fuente):
    if fuente == 'demo':
        from vega_datasets import data
        return data.stocks()
    p = Path(fuente)
    lector = {'.csv': pd.read_csv, '.xlsx': lambda x: pd.read_excel(x, header=1), '.xls': lambda x: pd.read_excel(x, header=1),
              '.parquet': pd.read_parquet, '.json': pd.read_json}[p.suffix.lower()]
    return lector(p)

df = cargar(FUENTE)
df.head()

,JJOO,AÑO,DEPORTE,DISCIPLINA,N°PRUEBAS HOMBRES,N° PRUEBAS MUJERES,N° PRUEBAS MIXTAS,N° PRUEBAS ABIERTAS,N° MUJERES,N° HOMBRES,LINK
0,París,2024,Fútbol,Fútbol,1,1,0,0,215,291,https://www.olympedia.org/editions/63/sports/FBL
1,París,2024,Atletismo,Atletismo,23,23,2,0,917,952,https://www.olympedia.org/editions/63/sports/ATH
2,París,2024,Deportes Acuáticos,Waterpolo,1,1,0,0,129,156,https://www.olympedia.org/editions/63/sports/WPO
3,París,2024,Deportes Acuáticos,Natación Artística,0,2,0,0,96,0,https://www.olympedia.org/editions/63/sports/SWA
4,París,2024,Deportes Acuáticos,Natación,17,17,1,0,390,463,https://www.olympedia.org/editions/63/sports/SWM


## 2. Exploración inicial

In [3]:
# Calidad de datos: nulos y duplicados
calidad = pd.DataFrame({
    'nulos': df.isna().sum(),
    '% nulos': (df.isna().mean() * 100).round(2),
    'únicos': df.nunique(),
    'tipo': df.dtypes.astype(str),
})
display(calidad)

,nulos,% nulos,únicos,tipo
JJOO,0,0.00,24,object
AÑO,0,0.00,29,int64
DEPORTE,0,0.00,45,object
DISCIPLINA,0,0.00,76,object
N°PRUEBAS HOMBRES,0,0.00,27,int64
N° PRUEBAS MUJERES,0,0.00,22,int64
N° PRUEBAS MIXTAS,0,0.00,4,int64
N° PRUEBAS ABIERTAS,0,0.00,11,int64
N° MUJERES,0,0.00,194,int64
N° HOMBRES,0,0.00,318,int64


### 2.1 Limpieza / tipos
Ajusta los nombres de columnas a los de tus datos (`COL_FECHA`, `COL_CAT`, `COL_VALOR`).

In [4]:
COL_FECHA, COL_CAT, COL_VALOR = 'AÑO', 'DEPORTE', 'N° MUJERES'   # Adaptado a tus columnas reales

# Convertir el año a formato datetime para que la escala de tiempo funcione
df[COL_FECHA] = pd.to_datetime(df[COL_FECHA].astype(str), format='%Y')

# Asegurar que el valor a graficar sea numérico
df[COL_VALOR] = pd.to_numeric(df[COL_VALOR], errors='coerce')

df = df.dropna(subset=[COL_FECHA, COL_CAT, COL_VALOR])
df = df.drop_duplicates().sort_values([COL_CAT, COL_FECHA]).reset_index(drop=True)

## 3. Gráfico principal: precio promedio por año y símbolo

In [5]:
import pandas as pd
import altair as alt

# 1. Definir las columnas de interés
columnas_pruebas = [
    'N°PRUEBAS HOMBRES',
    'N° PRUEBAS MUJERES',
    'N° PRUEBAS MIXTAS',
    'N° PRUEBAS ABIERTAS'
]

# 2. Asegurar que las columnas sean numéricas y rellenar nulos temporalmente con 0 para la agregación
df_pruebas = df.copy()
for col in columnas_pruebas:
    df_pruebas[col] = pd.to_numeric(df_pruebas[col], errors='coerce').fillna(0)

# 3. Agrupar la suma de pruebas por año
resumen_pruebas = df_pruebas.groupby(COL_FECHA, as_index=False)[columnas_pruebas].sum()

# 4. Transformar a formato largo (melt) para facilitar la graficación en Altair con colores
df_melted = resumen_pruebas.melt(
    id_vars=[COL_FECHA],
    value_vars=columnas_pruebas,
    var_name='Tipo de Prueba',
    value_name='Cantidad de Pruebas'
)

# 5. Crear el gráfico de líneas múltiples con puntos
grafico_lineas_pruebas = (
    alt.Chart(df_melted)
    .mark_line(point=True)
    .encode(
        x=alt.X(f'{COL_FECHA}:T', title='Año', axis=alt.Axis(format='%Y')),
        y=alt.Y('Cantidad de Pruebas:Q', title='Total de Pruebas'),
        color=alt.Color('Tipo de Prueba:N', title='Categoría de Prueba'),
        tooltip=[
            alt.Tooltip(f'year({COL_FECHA}):T', title='Año', format='%Y'),
            alt.Tooltip('Tipo de Prueba:N'),
            alt.Tooltip('Cantidad de Pruebas:Q', format=',.0f')
        ]
    )
    .properties(
        width=600,
        height=400,
        title='Evolución Histórica de los Tipos de Pruebas Olímpicas'
    )
    .interactive()
)

grafico_lineas_pruebas

alt.Chart(...)

### Versión interactiva (opcional): clic en la leyenda para resaltar un símbolo

## 4. Exportación

In [ ]:
# 4.1 Datos limpios
df.to_csv(OUT / 'datos_limpios.csv', index=False)
df.to_parquet(OUT / 'datos_limpios.parquet', index=False)

# 4.2 Tabla agregada que alimenta el gráfico
agg = (df.assign(anio=df[COL_FECHA].dt.year)
         .groupby([COL_CAT, 'anio'], as_index=False)[COL_VALOR].mean()
         .rename(columns={COL_VALOR: 'precio_promedio'}))
agg.to_csv(OUT / 'precio_promedio_anual.csv', index=False)

# 4.3 Resumen de calidad de datos
calidad.to_csv(OUT / 'calidad_datos.csv')

# 4.4 Gráficos
chart.save(OUT / 'grafico.json')          # especificación Vega-Lite
chart_int.save(OUT / 'grafico.html')      # interactivo
try:
    chart.save(OUT / 'grafico.png', scale_factor=2)   # requiere vl-convert-python
    chart.save(OUT / 'grafico.svg')
except Exception as e:
    print('PNG/SVG omitidos (instala vl-convert-python):', e)

sorted(p.name for p in OUT.iterdir())

['calidad_datos.csv',
 'datos_limpios.csv',
 'datos_limpios.parquet',
 'grafico.html',
 'grafico.json',
 'grafico.png',
 'grafico.svg',
 'precio_promedio_anual.csv']